![Header Image](../assets/header_image.png "Header Image")

# Devoir Optionnel 6 : URDF avec ROS 2 — Description de Robots

Bienvenue dans ce tutoriel sur l'URDF avec **ROS 2** !
L'URDF (Unified Robot Description Format) est le même format XML qu'en ROS 1,
mais les outils et commandes ROS 2 sont différents.

> **Pré-requis :** Avoir complété `3_introduction_to_ros2_fr.ipynb`.
> Sélectionnez le kernel **"Python 3.8 (ROS 2 Foxy)"**
> (**Kernel >> Change Kernel >> Python 3.8 (ROS 2 Foxy)**).

Dans ce devoir, vous allez

- **revoir la structure URDF** (liens, articulations, géométries) — identique en ROS 1 et ROS 2
- **générer un fichier URDF en Python** (approche ROS 2 recommandée vs copier-coller XML)
- **publier le `robot_description`** sur un topic ROS 2 avec `rclpy`
- **utiliser les commandes ROS 2** : `ros2 run robot_state_publisher`, `ros2 topic echo`
- **étendre le robot** en ajoutant un capteur caméra et un LiDAR via Python

# ROS 1 vs ROS 2 : URDF et Robot State Publisher

| Aspect | ROS 1 | ROS 2 |
|--------|--------|--------|
| **Format URDF** | XML (.urdf / .xacro) | **Identique** |
| **Visionneuse Jupyter** | `jupyterlab-urdf` | `jupyterlab-urdf` (identique) |
| **Publisher description** | `roslaunch robot_state_publisher ...` | `ros2 run robot_state_publisher robot_state_publisher` |
| **Topic description** | `/robot_description` (via paramètre) | `/robot_description` (topic `std_msgs/String`) |
| **Inspection topics** | `rostopic echo /robot_description` | `ros2 topic echo /robot_description` |
| **Xacro** | `rosrun xacro xacro fichier.xacro` | `ros2 run xacro xacro fichier.xacro` |
| **Visualisation 3D** | RViz | **RViz2** |

La bonne nouvelle : si vous connaissez l'URDF en ROS 1, vous pouvez réutiliser vos fichiers
directement en ROS 2 — seules les commandes changent.

# Rappel : Structure d'un fichier URDF

Un fichier URDF décrit un robot comme un **arbre de liens connectés par des articulations**.

## Les Liens (`<link>`)
Représentent les composants physiques du robot. Chaque lien peut avoir :
- `<visual>` : géométrie visible (boîte, cylindre, sphère, ou maillage `.dae`/`.stl`)
- `<collision>` : géométrie pour les calculs de collision physique
- `<inertial>` : masse et matrice d'inertie (pour la simulation dynamique)

## Les Articulations (`<joint>`)
Définissent la relation entre deux liens. Types principaux :

| Type | Description | Exemple |
|------|-------------|------|
| `fixed` | Connexion rigide, aucun mouvement | Capteur sur châssis |
| `continuous` | Rotation sans limite | Roue motrice |
| `revolute` | Rotation avec limites min/max | Bras robotique |
| `prismatique` | Translation linéaire avec limites | Ascenseur |

## Xacro
Extension XML permettant d'éviter la répétition : variables, macros, inclusion de fichiers.
En ROS 2 : `ros2 run xacro xacro robot.xacro`

# Configurer l'environnement

In [ ]:
!source /opt/ros/foxy/setup.bash

In [ ]:
import sys
sys.path.insert(0, '/opt/ros/foxy/lib/python3.8/site-packages/')

import platform
print("Python utilisé :", platform.python_version())
assert platform.python_version_tuple()[1] == '8', \
    "ERREUR : mauvais kernel ! Sélectionnez 'Python 3.8 (ROS 2 Foxy)' dans Kernel >> Change Kernel."
print("Kernel OK — Python 3.8 confirmé.")

In [ ]:
import rclpy
from rclpy.node import Node
from std_msgs.msg import String

import os
import threading
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

print("Bibliothèques importées avec succès !")

# Générer le fichier URDF avec Python

Au lieu de copier-coller du XML, nous allons **générer le fichier URDF programmatiquement**
avec Python. C'est l'approche moderne recommandée, surtout pour les robots complexes.
Elle permet de paramétrer les dimensions, couleurs et positions facilement.

Nous allons construire un robot mobile à deux roues avec :
1. `base_link` — lien racine (référentiel du robot)
2. `chassis` — carrosserie (boîte grise)
3. `left_wheel` / `right_wheel` — roues motrices (cylindres bleus)
4. `sphere_wheel` — roulette avant (sphère noire)
5. `lidar_link` — capteur LiDAR (cylindre violet)
6. `camera_link` — caméra avant (boîte rouge) — **ajout ROS 2**

In [ ]:
def generer_urdf(avec_camera=True):
    """
    Génère le contenu URDF d'un robot mobile 2 roues avec capteurs.
    Retourne une chaîne XML valide.
    """
    camera_xml = ""
    if avec_camera:
        camera_xml = """
    <!-- CAMÉRA AVANT (ajout ROS 2) -->
    <joint name="camera_joint" type="fixed">
        <parent link="chassis"/>
        <child link="camera_link"/>
        <origin xyz="0.27 0 0.12" rpy="0 0 0"/>
    </joint>
    <link name="camera_link">
        <visual>
            <geometry>
                <box size="0.04 0.08 0.04"/>
            </geometry>
            <material name="red">
                <color rgba="0.8 0.1 0.1 1"/>
            </material>
        </visual>
        <collision>
            <geometry>
                <box size="0.04 0.08 0.04"/>
            </geometry>
        </collision>
    </link>"""

    return f"""<?xml version="1.0"?>
<robot xmlns:xacro="http://www.ros.org/wiki/xacro" name="robot_ros2">

    <!-- LIEN RACINE -->
    <link name="base_link"/>

    <!-- CHÂSSIS -->
    <joint name="chassis_joint" type="fixed">
        <parent link="base_link"/>
        <child link="chassis"/>
        <origin xyz="-0.1 0 0"/>
    </joint>
    <link name="chassis">
        <visual>
            <origin xyz="0.15 0 0.075" rpy="0 0 0"/>
            <geometry>
                <box size="0.3 0.3 0.15"/>
            </geometry>
            <material name="grey">
                <color rgba="0.5 0.5 0.5 1"/>
            </material>
        </visual>
        <collision>
            <origin xyz="0.15 0 0.075" rpy="0 0 0"/>
            <geometry>
                <box size="0.3 0.3 0.15"/>
            </geometry>
        </collision>
    </link>

    <!-- ROUE GAUCHE -->
    <joint name="left_wheel_joint" type="continuous">
        <parent link="base_link"/>
        <child link="left_wheel"/>
        <origin xyz="0 0.175 0" rpy="-1.5707963 0 0"/>
        <axis xyz="0 0 1"/>
    </joint>
    <link name="left_wheel">
        <visual>
            <geometry>
                <cylinder length="0.04" radius="0.05"/>
            </geometry>
            <material name="blue">
                <color rgba="0.1 0.3 0.8 1"/>
            </material>
        </visual>
        <collision>
            <geometry>
                <cylinder length="0.04" radius="0.05"/>
            </geometry>
        </collision>
    </link>

    <!-- ROUE DROITE -->
    <joint name="right_wheel_joint" type="continuous">
        <parent link="base_link"/>
        <child link="right_wheel"/>
        <origin xyz="0 -0.175 0" rpy="-1.5707963 0 0"/>
        <axis xyz="0 0 1"/>
    </joint>
    <link name="right_wheel">
        <visual>
            <geometry>
                <cylinder length="0.04" radius="0.05"/>
            </geometry>
            <material name="blue">
                <color rgba="0.1 0.3 0.8 1"/>
            </material>
        </visual>
        <collision>
            <geometry>
                <cylinder length="0.04" radius="0.05"/>
            </geometry>
        </collision>
    </link>

    <!-- ROULETTE AVANT -->
    <joint name="sphere_wheel_joint" type="fixed">
        <parent link="chassis"/>
        <child link="sphere_wheel"/>
        <origin xyz="0.24 0 0" rpy="0 0 0"/>
    </joint>
    <link name="sphere_wheel">
        <visual>
            <geometry>
                <sphere radius="0.05"/>
            </geometry>
            <material name="black">
                <color rgba="0.1 0.1 0.1 1"/>
            </material>
        </visual>
        <collision>
            <geometry>
                <sphere radius="0.05"/>
            </geometry>
        </collision>
    </link>

    <!-- LIDAR -->
    <joint name="lidar_joint" type="fixed">
        <parent link="chassis"/>
        <child link="lidar_link"/>
        <origin xyz="0.15 0 0.175" rpy="0 0 0"/>
    </joint>
    <link name="lidar_link">
        <visual>
            <geometry>
                <cylinder length="0.04" radius="0.04"/>
            </geometry>
            <material name="purple">
                <color rgba="0.6 0.0 0.8 1"/>
            </material>
        </visual>
        <collision>
            <geometry>
                <cylinder length="0.04" radius="0.04"/>
            </geometry>
        </collision>
    </link>
{camera_xml}
</robot>
"""

urdf_contenu = generer_urdf(avec_camera=True)
print("URDF généré avec succès !")
print(f"Taille : {len(urdf_contenu)} caractères")
print("Liens présents :", urdf_contenu.count('<link name='))

# Écrire le fichier URDF sur le disque

Nous sauvegardons le fichier URDF généré dans le dossier du cours.
Vous pouvez ensuite **double-cliquer sur le fichier dans JupyterLab**
pour l'ouvrir dans la visionneuse `jupyterlab-urdf`.

In [ ]:
urdf_path = '/home/jovyan/tp-va/section_1_introduction_and_tools/robot_ros2.urdf'

with open(urdf_path, 'w') as f:
    f.write(urdf_contenu)

print(f"Fichier URDF écrit : {urdf_path}")
print()
print("Pour le visualiser :")
print("  1. Dans le panneau de fichiers JupyterLab (à gauche)")
print("  2. Naviguez vers section_1_introduction_and_tools/")
print("  3. Double-cliquez sur 'robot_ros2.urdf'")
print("  => La visionneuse jupyterlab-urdf s'ouvre automatiquement")

# Afficher le contenu URDF

In [ ]:
print(urdf_contenu)

# Publier le robot_description avec ROS 2

En ROS 2, le `robot_state_publisher` lit le fichier URDF et publie :
- Le contenu URDF sur `/robot_description` (type `std_msgs/String`)
- Les transformations TF2 sur `/tf` et `/tf_static`

Nous pouvons aussi publier directement depuis Python, ce qui est utile pour tester.

In [ ]:
rclpy.init()
node = rclpy.create_node('urdf_publisher')
spin_thread = threading.Thread(target=rclpy.spin, args=(node,), daemon=True)
spin_thread.start()

# Publier le robot_description comme un message String
pub_desc = node.create_publisher(String, '/robot_description', 10)

# Attendre que le subscriber soit connecté
time.sleep(0.5)

msg = String()
msg.data = urdf_contenu
pub_desc.publish(msg)

print("robot_description publié sur /robot_description")
print(f"Taille du message : {len(msg.data)} caractères")

# Recevoir et valider le robot_description

In [ ]:
desc_recu = [None]

def callback_description(msg):
    desc_recu[0] = msg.data
    n_links = msg.data.count('<link name=')
    n_joints = msg.data.count('<joint name=')
    print(f"robot_description reçu : {n_links} liens, {n_joints} articulations")

sub_desc = node.create_subscription(String, '/robot_description', callback_description, 10)

# Re-publier pour déclencher le callback
time.sleep(0.2)
pub_desc.publish(msg)
time.sleep(0.5)

if desc_recu[0]:
    print("Validation OK — URDF correctement transmis via ROS 2 !")
else:
    print("Aucun message reçu — ré-exécutez cette cellule.")

# Utiliser robot_state_publisher depuis le Terminal

Le `robot_state_publisher` est le nœud officiel ROS 2 pour diffuser le modèle du robot.
Ouvrez un terminal (**Fichier >> Nouveau >> Terminal**) et exécutez :

```bash
source /opt/ros/foxy/setup.bash

# Lancer robot_state_publisher avec le fichier URDF généré
ros2 run robot_state_publisher robot_state_publisher \
    --ros-args -p robot_description:="$(cat /home/jovyan/tp-va/section_1_introduction_and_tools/robot_ros2.urdf)"
```

Dans un second terminal :

```bash
source /opt/ros/foxy/setup.bash

# Voir les topics publiés par robot_state_publisher
ros2 topic list
# Devrait afficher : /robot_description, /tf, /tf_static, /parameter_events, /rosout

# Lire le robot_description publié
ros2 topic echo /robot_description --no-arr

# Voir les informations du topic
ros2 topic info /robot_description

# Voir l'arbre TF2 (liens et transformations)
ros2 run tf2_tools view_frames.py
```

> **Différence clé ROS 1 vs ROS 2 :**
> - ROS 1 : `roslaunch robot_state_publisher robot_state_publisher.launch`
> - ROS 2 : `ros2 run robot_state_publisher robot_state_publisher --ros-args -p robot_description:=...`

# Visualisation 3D Schématique du Robot avec matplotlib

Puisque nous n'avons pas RViz2 dans ce conteneur, nous allons créer une **représentation
schématique 3D** du robot avec `matplotlib` pour visualiser l'arbre de liens.

In [ ]:
def dessiner_boite(ax, centre, taille, couleur, alpha=0.4):
    """Dessine une boîte 3D."""
    cx, cy, cz = centre
    lx, ly, lz = [s/2 for s in taille]
    vertices = np.array([
        [cx-lx, cy-ly, cz-lz], [cx+lx, cy-ly, cz-lz],
        [cx+lx, cy+ly, cz-lz], [cx-lx, cy+ly, cz-lz],
        [cx-lx, cy-ly, cz+lz], [cx+lx, cy-ly, cz+lz],
        [cx+lx, cy+ly, cz+lz], [cx-lx, cy+ly, cz+lz],
    ])
    faces = [[vertices[j] for j in [0,1,2,3]], [vertices[j] for j in [4,5,6,7]],
             [vertices[j] for j in [0,1,5,4]], [vertices[j] for j in [2,3,7,6]],
             [vertices[j] for j in [1,2,6,5]], [vertices[j] for j in [4,7,3,0]]]
    poly = Poly3DCollection(faces, alpha=alpha, facecolor=couleur, edgecolor='black', linewidth=0.5)
    ax.add_collection3d(poly)

def dessiner_cylindre(ax, centre, rayon, hauteur, couleur, alpha=0.5, axe='z'):
    """Dessine un cylindre 3D."""
    cx, cy, cz = centre
    theta = np.linspace(0, 2*np.pi, 20)
    if axe == 'z':
        X = cx + rayon * np.cos(theta)
        Y = cy + rayon * np.sin(theta)
        for z0 in [cz - hauteur/2, cz + hauteur/2]:
            ax.plot(X, Y, z0 * np.ones_like(X), color=couleur, linewidth=1)
        for th in theta[::4]:
            ax.plot([cx + rayon*np.cos(th)]*2, [cy + rayon*np.sin(th)]*2,
                    [cz - hauteur/2, cz + hauteur/2], color=couleur, linewidth=0.5, alpha=alpha)
    elif axe == 'y':
        Z = cz + rayon * np.cos(theta)
        X = cx + rayon * np.sin(theta)
        for y0 in [cy - hauteur/2, cy + hauteur/2]:
            ax.plot(X, y0 * np.ones_like(X), Z, color=couleur, linewidth=1)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

# Châssis (boîte grise)
dessiner_boite(ax, centre=(0.05, 0, 0.075), taille=(0.3, 0.3, 0.15), couleur='lightgray')

# Roue gauche (cylindre bleu, axe Y)
dessiner_cylindre(ax, centre=(0, 0.195, 0), rayon=0.05, hauteur=0.04, couleur='steelblue', axe='y')

# Roue droite (cylindre bleu, axe Y)
dessiner_cylindre(ax, centre=(0, -0.195, 0), rayon=0.05, hauteur=0.04, couleur='steelblue', axe='y')

# Roulette avant (sphère noire)
u, v = np.mgrid[0:2*np.pi:15j, 0:np.pi:10j]
r = 0.05
ax.plot_wireframe(0.29 + r*np.cos(u)*np.sin(v),
                  r*np.sin(u)*np.sin(v),
                  r*np.cos(v), color='black', linewidth=0.5, alpha=0.5)

# LiDAR (cylindre violet)
dessiner_cylindre(ax, centre=(0.05+0.1, 0, 0.15+0.02), rayon=0.04, hauteur=0.04, couleur='purple', axe='z')

# Caméra (boîte rouge)
dessiner_boite(ax, centre=(0.05+0.17, 0, 0.075+0.045), taille=(0.04, 0.08, 0.04), couleur='red', alpha=0.6)

# Annotations des liens
labels = [
    (0.05, 0, 0.075, 'chassis', 'gray'),
    (0, 0.22, 0, 'left_wheel', 'steelblue'),
    (0, -0.22, 0, 'right_wheel', 'steelblue'),
    (0.34, 0, 0.05, 'sphere_wheel', 'black'),
    (0.15, 0, 0.2, 'lidar_link', 'purple'),
    (0.22, 0, 0.16, 'camera_link', 'red'),
]
for x, y, z, label, col in labels:
    ax.text(x, y, z + 0.05, label, fontsize=7, color=col, ha='center')

ax.set_xlim(-0.2, 0.5)
ax.set_ylim(-0.3, 0.3)
ax.set_zlim(-0.1, 0.3)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_zlabel('z (m)')
ax.set_title('Représentation schématique 3D du robot URDF\n(châssis + 2 roues + LiDAR + caméra)')
ax.view_init(elev=25, azim=-45)

plt.tight_layout()
plt.show()

# Analyser la Structure de l'Arbre URDF avec Python

In [ ]:
import xml.etree.ElementTree as ET

root = ET.fromstring(urdf_contenu)

print(f"Nom du robot : {root.get('name')}")
print()

liens = root.findall('link')
print(f"Liens ({len(liens)}) :")
for lien in liens:
    visuel = lien.find('visual')
    if visuel is not None:
        geom = visuel.find('geometry')
        type_geom = list(geom)[0].tag if geom is not None and len(list(geom)) > 0 else 'inconnu'
    else:
        type_geom = '(aucune géométrie)'
    print(f"  - {lien.get('name'):20s} | géométrie: {type_geom}")

print()
articulations = root.findall('joint')
print(f"Articulations ({len(articulations)}) :")
for jnt in articulations:
    parent = jnt.find('parent').get('link')
    child  = jnt.find('child').get('link')
    type_j = jnt.get('type')
    print(f"  - {jnt.get('name'):25s} | {parent} --> {child} [{type_j}]")

# Exercice : Ajouter un Nouveau Capteur au Robot

Modifiez la fonction `generer_urdf()` pour ajouter un **capteur de profondeur** (Depth Camera)
sur le dessus du châssis, orienté vers le haut :

- **Lien** : `depth_camera_link`
- **Géométrie** : boîte `0.05 x 0.05 x 0.03`
- **Couleur** : vert (`rgba="0.1 0.8 0.1 1"`)
- **Position** : `xyz="0.05 0 0.165"` (sur le dessus du châssis)
- **Type d'articulation** : `fixed` (rattaché au `chassis`)

Après avoir modifié la fonction, exécutez à nouveau les cellules pour :
1. Générer le nouveau fichier URDF
2. L'écrire sur le disque
3. Vérifier qu'il apparaît dans l'arbre (7 liens au lieu de 6)
4. L'ouvrir dans la visionneuse `jupyterlab-urdf`

In [ ]:
# Vérifier que le fichier URDF a bien été créé et est lisible par ROS 2
import subprocess

resultat = subprocess.run(
    ['python3', '-c',
     f'import xml.etree.ElementTree as ET; '
     f'tree = ET.parse("{urdf_path}"); '
     f'root = tree.getroot(); '
     f'print(f"URDF valide : {{root.get(\"name\")}} avec {{len(root.findall(\"link\"))}} liens")'
    ],
    capture_output=True, text=True
)
print(resultat.stdout.strip())
if resultat.returncode == 0:
    print("Le fichier URDF est syntaxiquement valide.")
else:
    print("Erreur :", resultat.stderr)

# Utiliser la Visionneuse jupyterlab-urdf

La bibliothèque `jupyterlab-urdf` est installée dans ce conteneur et fonctionne **identiquement
en ROS 1 et ROS 2**, puisqu'elle ne dépend pas de ROS directement.

## Ouvrir le fichier URDF :

1. Dans le panneau de fichiers à gauche, naviguez vers `section_1_introduction_and_tools/`
2. **Double-cliquez** sur `robot_ros2.urdf` → la visionneuse 3D s'ouvre

## Modifier le fichier et voir les changements en temps réel :

1. Clic droit sur `robot_ros2.urdf` → **Ouvrir avec → Éditeur**
2. Modifiez, par exemple, la taille du châssis (`box size`)
3. Sauvegardez (Ctrl+S)
4. La visionneuse se met à jour automatiquement

## Créer un nouveau fichier URDF depuis JupyterLab :

**Fichier >> Nouveau >> Créer un nouveau URDF** (si l'extension est activée dans le lanceur)

> **Note :** La visionneuse affiche la géométrie visuelle. Pour la simulation physique
> (Gazebo), les balises `<collision>` et `<inertial>` sont également nécessaires.

# Arrêter le nœud ROS 2

In [ ]:
node.destroy_node()
rclpy.shutdown()
print("Nœud ROS 2 arrêté proprement.")

# Résumé

- Vous avez revu la **structure URDF** (liens, articulations, géométries) — identique en ROS 1 et ROS 2.
- Vous avez appris à **générer un fichier URDF programmatiquement en Python**, une approche plus flexible que le copier-coller XML.
- Vous avez **publié le `robot_description`** sur un topic ROS 2 avec `rclpy`, sans master ROS.
- Vous avez **analysé la structure de l'arbre URDF** avec le module `xml.etree.ElementTree`.
- Vous avez appris les **commandes ROS 2** spécifiques à URDF : `ros2 run robot_state_publisher`, `ros2 topic echo /robot_description`.
- Vous avez créé une **visualisation schématique 3D** du robot avec matplotlib.
- La **visionneuse `jupyterlab-urdf`** fonctionne identiquement en ROS 1 et ROS 2.